# MLOps：TVに対応する世帯でのF1の有無を予測する

5局の同じ合成TVデータを使い、世帯にF1（女性20〜34歳）がいるかを予測します。20,000台のうち正解が公開される2,000台から学びます。

静的な世帯属性の推定です。今誰が見ているか、実在の人の属性、人数は特定しません。確率は未校正で、人数や世帯の確定数として合計できません。

特徴量 → 学習・評価 → Model Registry → 登録モデルで確率を計算 → 保存・照合 → サービス停止の順に、コードセルを1つずつ実行してください。詳しくは[第3章](../docs/03_mlops.md)を参照します。

## 1. 実行前の確認

第2章の6回のdbt buildを完了してから実行します。SnowsightのWorkspace NotebookでPythonランタイムを選び、`BCAST_PLATFORM_ENGINEER_ROLE`、`BCAST_PLATFORM_COMMON_WH`を使います。必要なパッケージは pandas、scikit-learn、snowflake-ml-python、snowflake-snowpark-pythonです。

このNotebookはローカルPCでは実行しません。モデルV1が既にある場合は登録前に停止します。別版を試す場合はMODEL_VERSIONをV2などに変更して上から実行します。既存モデルを削除しません。予測結果テーブルは最後の保存セルで置き換えます。

In [ ]:
import re
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_score, recall_score, confusion_matrix
from sklearn.inspection import permutation_importance
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as sf
from snowflake.snowpark.types import StructType, StructField, StringType, LongType, DoubleType
from snowflake.ml.registry import Registry
from snowflake.ml.model import model_signature

session = get_active_session()
if session.get_current_role().strip('"') != 'BCAST_PLATFORM_ENGINEER_ROLE':
    raise RuntimeError('NotebookのロールをBCAST_PLATFORM_ENGINEER_ROLEに変更してください。')
session.use_warehouse('BCAST_PLATFORM_COMMON_WH')
session.use_database('BCAST_PLATFORM_HANDSON')
session.use_schema('ML')
MODEL_NAME = 'TV_F1_PRESENCE_MODEL'
MODEL_VERSION = 'V1'
THRESHOLD = 0.5
EXPECTED_DEVICES = {f'C{number:06d}' for number in range(1, 20001)}
GENRES = ['NEWS', 'DRAMA', 'VARIETY', 'ANIME', 'SPORTS', 'MUSIC', 'MOVIE', 'INFO']
FEATURE_COLUMNS = [genre + '_SHARE' for genre in GENRES] + ['TOTAL_MINUTES', 'TOTAL_SESSIONS', 'ACTIVE_DAYS', 'MEAN_MINUTES']
prediction_model = None
registered_version = None
if not re.fullmatch(r'V[1-9][0-9]*', MODEL_VERSION):
    raise ValueError('MODEL_VERSIONはV1、V2などを指定します。')
print('scikit-learn', sklearn.__version__)
print('登録予定:', MODEL_NAME, MODEL_VERSION)
print('F1は女性20〜34歳の区分です。予測対象は架空のTVに対応する世帯での有無で、現在の視聴者や人数ではありません。')

## 2. 1台につき1行のヒントを作る

5局の共通マートから、8ジャンルの時間割合、総視聴分数、総視聴回数、視聴日数、1回の平均分数を作ります。12項目を特徴量と呼びます。

集計は共通WHで行い、Notebookへ取得する上限は20,001行です。20,000台ちょうどであることを確認します。約1,354万行の分展開データをPythonへ持ち込みません。

端末ID、正解公開フラグ、正解TARGET_F1は特徴量に使いません。非パネル18,000台のNULLは不明であり、0ではありません。

In [ ]:
prediction_model = None
daily = session.table('BCAST_PLATFORM_HANDSON.COMMON.VIEWING_DAILY')
features_sdf = daily.group_by('DEVICE_ID').agg(
    sf.sum('VIEW_MINUTES').cast('double').alias('TOTAL_MINUTES'),
    sf.sum('SESSION_COUNT').cast('double').alias('TOTAL_SESSIONS'),
    sf.count_distinct('VIEW_DATE').cast('double').alias('ACTIVE_DAYS'),
    *[sf.sum(sf.when(sf.col('GENRE') == genre, sf.col('VIEW_MINUTES')).otherwise(sf.lit(0))).cast('double').alias(genre + '_MINUTES') for genre in GENRES]
)
for genre in GENRES:
    features_sdf = features_sdf.with_column(genre + '_SHARE', sf.col(genre + '_MINUTES') / sf.nullif(sf.col('TOTAL_MINUTES'), sf.lit(0)))
features_sdf = features_sdf.with_column('MEAN_MINUTES', sf.col('TOTAL_MINUTES') / sf.nullif(sf.col('TOTAL_SESSIONS'), sf.lit(0)))
features_pdf = features_sdf.select('DEVICE_ID', *FEATURE_COLUMNS).sort('DEVICE_ID').limit(20001).to_pandas()
labels_pdf = session.table('BCAST_PLATFORM_HANDSON.RAW.DEVICE_LABELS').select('DEVICE_ID', 'LABEL_AVAILABLE', 'TARGET_F1').sort('DEVICE_ID').limit(20001).to_pandas()
for frame in [features_pdf, labels_pdf]:
    if len(frame) != 20000 or frame.DEVICE_ID.isna().any() or frame.DEVICE_ID.duplicated().any() or set(frame.DEVICE_ID) != EXPECTED_DEVICES:
        raise ValueError('欠損・重複なしのC000001〜C020000が必要です。第1・2章を確認してください。')
features_pdf[FEATURE_COLUMNS] = features_pdf[FEATURE_COLUMNS].astype('float64')
if not np.isfinite(features_pdf[FEATURE_COLUMNS].to_numpy()).all():
    raise ValueError('特徴量に欠損または無限大があります。')
if not np.allclose(features_pdf[[genre + '_SHARE' for genre in GENRES]].sum(axis=1), 1):
    raise ValueError('ジャンル割合の合計が1ではありません。')
if labels_pdf.LABEL_AVAILABLE.isna().any():
    raise ValueError('ラベル公開フラグに欠損があります。')
known_mask = labels_pdf.LABEL_AVAILABLE.eq(True)
if known_mask.sum() != 2000 or labels_pdf.loc[~known_mask, 'TARGET_F1'].notna().any():
    raise ValueError('正解公開2000台・不明18000台の契約に一致しません。')
known_pdf = features_pdf.merge(labels_pdf.loc[known_mask, ['DEVICE_ID', 'TARGET_F1']], on='DEVICE_ID', validate='one_to_one').sort_values('DEVICE_ID').reset_index(drop=True)
if known_pdf.TARGET_F1.isna().any() or not known_pdf.TARGET_F1.isin([0, 1]).all():
    raise ValueError('公開正解は0/1が必要です。NULLを0で埋めないでください。')
print('全TV:', len(features_pdf), '正解公開:', len(known_pdf))
print(known_pdf.TARGET_F1.value_counts().sort_index())
features_pdf.head()

## 3. 学習1,600台・採点400台で比較する

同じTVが学習と採点に入らないように分けます。F1あり・なしの比率も保ちます。設定と閾値0.5は先に固定し、採点データに合わせて変更しません。

HistGradientBoostingは複数の決定木を組み合わせる分類モデルです。「全員に学習データのF1割合を返す」単純な基準と比較します。

ROC AUCは順位付け（基準0.5）、Average Precisionは陽性を上位に集める度合い、Brierは確率の誤差（小さい方が良い）です。precisionは陽性とした中の正解率、recallは実際の陽性を見つけた割合です。混同行列で見逃しも確認します。

固定データのローカル検証ではAUC約0.608、Brierは基準より悪く、閾値0.5では陽性69台中1台しか検出していません。高精度モデル完成の例ではなく、評価から不足を発見する結果です。実環境の出力を確認し、性能を誇張しないでください。

In [ ]:
prediction_model = None
train_pdf, evaluation_pdf = train_test_split(known_pdf, test_size=0.2, random_state=42, stratify=known_pdf.TARGET_F1)
assert len(train_pdf) == 1600 and len(evaluation_pdf) == 400
assert set(train_pdf.DEVICE_ID).isdisjoint(evaluation_pdf.DEVICE_ID)
train_features = train_pdf[FEATURE_COLUMNS]
evaluation_features = evaluation_pdf[FEATURE_COLUMNS]
train_labels = train_pdf.TARGET_F1.astype('int64')
evaluation_labels = evaluation_pdf.TARGET_F1.astype('int64')
model = HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=15, l2_regularization=10, random_state=42)
model.fit(train_features, train_labels)
np.testing.assert_array_equal(model.classes_, [0, 1])
baseline = DummyClassifier(strategy='prior').fit(train_features, train_labels)
metrics_rows = []
for name, estimator in [('Prior baseline', baseline), ('HistGradientBoosting', model)]:
    probability = estimator.predict_proba(evaluation_features)[:, 1]
    predicted = (probability >= THRESHOLD).astype('int64')
    metrics_rows.append({'MODEL': name, 'ROC_AUC': roc_auc_score(evaluation_labels, probability), 'AVERAGE_PRECISION': average_precision_score(evaluation_labels, probability), 'BRIER': brier_score_loss(evaluation_labels, probability), 'PRECISION': precision_score(evaluation_labels, predicted, zero_division=0), 'RECALL': recall_score(evaluation_labels, predicted, zero_division=0)})
    print(name, '混同行列（行=正解0/1、列=予測0/1）:')
    print(confusion_matrix(evaluation_labels, predicted, labels=[0, 1]))
metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string(index=False))
print('Brierは小さいほど良い指標です。基準より悪ければ、確率の改善が必要です。閾値0.5は採点前に固定しました。')
importance = permutation_importance(model, evaluation_features, evaluation_labels, scoring='roc_auc', n_repeats=3, random_state=42, n_jobs=1)
print(pd.DataFrame({'FEATURE': FEATURE_COLUMNS, 'IMPORTANCE': importance.importances_mean}).sort_values('IMPORTANCE', ascending=False).to_string(index=False))
print('重要度はこの評価標本での手掛かりで、因果関係ではありません。これを見て同じ採点データに合わせた調整はしません。')

## 4. モデルに名前と版を付けて登録する

この後はSnowflakeにモデルを書き込みます。採点したモデルそのものを登録し、全件で学習し直しません。既存の同名・同版がある場合は停止します。続けるにはMODEL_VERSIONを別の版へ変更し、先頭から実行してください。

登録処理や依存パッケージ解決が失敗した場合は停止し、講師に相談してください。コンテナで学習しても、予測の実行先は明示的にWAREHOUSEを指定します。

In [ ]:
prediction_model = None
registered_version = None
registry = Registry(session=session, database_name='BCAST_PLATFORM_HANDSON', schema_name='ML')
existing_models = registry.show_models()
existing_models.columns = [str(column).upper() for column in existing_models.columns]
if not existing_models.empty:
    if 'NAME' not in existing_models.columns:
        raise RuntimeError('モデル一覧の列を確認してください。安全のため停止します。')
    if existing_models['NAME'].str.upper().eq(MODEL_NAME).any():
        existing_versions = registry.get_model(MODEL_NAME).show_versions()
        existing_versions.columns = [str(column).upper() for column in existing_versions.columns]
        if 'NAME' not in existing_versions.columns:
            raise RuntimeError('モデル版一覧の列を確認してください。安全のため停止します。')
        if existing_versions['NAME'].str.upper().eq(MODEL_VERSION).any():
            raise RuntimeError('同じ版が存在します。MODEL_VERSIONを別の版へ変更して先頭から実行してください。')
np.testing.assert_array_equal(model.classes_, [0, 1])
probability_signature = model_signature.infer_signature(train_features.head(10), model.predict_proba(train_features.head(10)), output_feature_names=['PROB_NO_F1', 'PROB_F1'])
registered_version = registry.log_model(
    model,
    model_name=MODEL_NAME,
    version_name=MODEL_VERSION,
    signatures={'predict_proba': probability_signature},
    conda_dependencies=['scikit-learn==' + sklearn.__version__],
    target_platforms=['WAREHOUSE'],
    options={'relax_version': False},
    metrics={column.lower(): float(metrics_df.iloc[1][column]) for column in ['ROC_AUC', 'AVERAGE_PRECISION', 'BRIER', 'PRECISION', 'RECALL']},
    comment='Synthetic household F1 presence. Uncalibrated probability, fixed threshold0.5. Not current viewer identity or person count.'
)
registered_name = registered_version.model_name
registered_version_name = registered_version.version_name
print('登録:', registered_name, registered_version_name)
registered_version.show_functions()

## 5. 登録したモデルで20,000台の確率を計算する

登録したpredict_probaを共通WHで実行します。登録時にクラス0をPROB_NO_F1、クラス1をPROB_F1へ明示的に対応付けています。実際の返却列が違う場合は停止します。

戻った行順で端末IDを付けず、保持された特徴量で照合します。同じ特徴量のTVは同じ確率になります。有限の0〜1、両クラスの合計1、20,000台への対応を確認します。

学習用TVも含む全台への付与ですが、ここで精度を採点しません。最終評価は先ほど分けた400台だけです。確率は未校正であり、実際のF1在籍や視聴者を確定しません。

In [ ]:
prediction_model = None
prediction_input = features_pdf[FEATURE_COLUMNS].drop_duplicates().reset_index(drop=True)
prediction_schema = StructType([StructField(column, DoubleType()) for column in FEATURE_COLUMNS])
prediction_sdf = session.create_dataframe(prediction_input.to_numpy().tolist(), schema=prediction_schema)
print(registered_version.show_functions())
registry_output = registered_version.run(prediction_sdf, function_name='predict_proba').limit(20001).to_pandas()
registry_output.columns = [str(column).upper() for column in registry_output.columns]
expected_columns = set(FEATURE_COLUMNS) | {'PROB_NO_F1', 'PROB_F1'}
if set(registry_output.columns) != expected_columns:
    raise RuntimeError('推論列が登録した署名と一致しません。列名を推測して保存しません: ' + str(registry_output.columns.tolist()))
if len(registry_output) != len(prediction_input) or registry_output.duplicated(FEATURE_COLUMNS).any():
    raise ValueError('推論行数または特徴量の一意性が一致しません。')
probabilities = registry_output[['PROB_NO_F1', 'PROB_F1']].astype('float64')
if not np.isfinite(probabilities.to_numpy()).all() or not probabilities.ge(0).all().all() or not probabilities.le(1).all().all() or not np.allclose(probabilities.sum(axis=1), 1):
    raise ValueError('確率は有限の0〜1で、両クラスの合計が1である必要があります。')
prediction_pdf = features_pdf.merge(registry_output, on=FEATURE_COLUMNS, how='left', validate='many_to_one')
if len(prediction_pdf) != 20000 or prediction_pdf.PROB_F1.isna().any():
    raise ValueError('全TVへの予測対応を確認できません。')
prediction_pdf['PREDICTED_HAS_F1'] = (prediction_pdf.PROB_F1 >= THRESHOLD).astype('int64')
prediction_model = (registered_version.model_name, registered_version.version_name)
prediction_pdf[['DEVICE_ID', 'PROB_F1', 'PREDICTED_HAS_F1']].head()

## 6. 確率・判定・モデル版を保存する

次のセルはML.PREDICTIONSを置き換えます。SAVE_RESULTS=Falseでは意図的に停止します。今回の予測で置き換えてよいと確認したときだけTrueへ変更します。

PROB_F1は未校正の確率、PREDICTED_HAS_F1は閾値0.5による予測ラベルです。PREDICTED_ATはUTCの保存時刻です。モデル登録はこの確認より前に実行されています。

20,000台を読み戻して照合します。保存後の確認に失敗しても書込みは自動取消されないので、次章へ進まず講師へ確認してください。

In [ ]:
SAVE_RESULTS = False
if not SAVE_RESULTS:
    raise RuntimeError('結果を置き換えてよい場合はSAVE_RESULTSをTrueに変更してください。')
if globals().get('prediction_model') != (registered_name, registered_version_name) or (MODEL_NAME, MODEL_VERSION) != (registered_name, registered_version_name):
    raise ValueError('保存前確認: このモデル版で推論を完了してください。')
if len(prediction_pdf) != 20000 or prediction_pdf.DEVICE_ID.duplicated().any() or set(prediction_pdf.DEVICE_ID) != EXPECTED_DEVICES:
    raise ValueError('保存前確認: 全20000台のIDが一致しません。')
probability = prediction_pdf.PROB_F1.astype('float64')
if not np.isfinite(probability).all() or not probability.between(0, 1).all() or not prediction_pdf.PREDICTED_HAS_F1.eq((probability >= 0.5).astype('int64')).all():
    raise ValueError('保存前確認: 確率と閾値0.5によるラベルが一致しません。')
if registered_name != registered_version.model_name or registered_version_name != registered_version.version_name:
    raise ValueError('保存前確認: 登録モデルの版が一致しません。')
output_rows = [(str(row.DEVICE_ID), float(row.PROB_F1), int(row.PREDICTED_HAS_F1)) for row in prediction_pdf.itertuples(index=False)]
output_schema = StructType([StructField('DEVICE_ID', StringType()), StructField('PROB_F1', DoubleType()), StructField('PREDICTED_HAS_F1', LongType())])
output_sdf = session.create_dataframe(output_rows, schema=output_schema).with_column('MODEL_NAME', sf.lit(registered_name)).with_column('MODEL_VERSION', sf.lit(registered_version_name)).with_column('PREDICTED_AT', sf.convert_timezone(sf.lit('UTC'), sf.current_timestamp()).cast('timestamp_ntz'))
output_sdf.write.mode('overwrite').save_as_table('BCAST_PLATFORM_HANDSON.ML.PREDICTIONS')
saved_pdf = session.table('BCAST_PLATFORM_HANDSON.ML.PREDICTIONS').limit(20001).to_pandas()
if len(saved_pdf) != 20000 or saved_pdf.DEVICE_ID.duplicated().any() or set(saved_pdf.DEVICE_ID) != EXPECTED_DEVICES:
    raise RuntimeError('保存後の端末照合に失敗しました。後続へ進まないでください。')
compare_columns = ['DEVICE_ID', 'PROB_F1', 'PREDICTED_HAS_F1']
pd.testing.assert_frame_equal(prediction_pdf[compare_columns].sort_values('DEVICE_ID').reset_index(drop=True), saved_pdf[compare_columns].sort_values('DEVICE_ID').reset_index(drop=True), check_dtype=False, rtol=1e-12, atol=1e-12)
if not saved_pdf.MODEL_NAME.eq(registered_name).all() or not saved_pdf.MODEL_VERSION.eq(registered_version_name).all() or saved_pdf.PREDICTED_AT.isna().any():
    raise RuntimeError('保存後のモデル版またはUTC保存時刻が一致しません。')
print('20000台のF1予測を保存・照合しました。評価値とモデル版を記録してから、Connected → サービス名 → SuspendでSUSPENDEDを確認してください。')

## 7. できたこと・やっていないこと

- 学習と採点を分け、比較用の基準とともにモデルを評価しました。
- 名前と版を付けてモデルを登録し、その登録モデルを使って予測しました。
- 結果にモデル版を残しました。これが今回のMLOpsの入口です。
- 自動再学習、ドリフト監視、本番精度の保証はこのNotebookに含みません。
- 実行中にパッケージ・権限・戻り値のエラーが出た場合は、後続セルを飛ばして進めず講師へ報告します。
- 保存・照合まで成功したら、評価値・モデル版・必要な出力を記録します。
- 続いて **Connected → サービス名 → Suspend** で自分のNotebookサービスを停止し、**SUSPENDED** を確認します。ブラウザーを閉じるだけでは停止しません。同じサービスにつながる別のNotebookも切断され、変数や追加パッケージが失われるため、別作業が動いていないことを先に確認します。
- 詳細は[第3章の停止手順](../docs/03_mlops.md#8-notebookの実行サービスを停止する)を確認してください。共有compute poolは停止・削除しません。
- サービス停止を確認してから、第4章のStreamlitで保存済みの結果を確認します。